# 05 — Atribución de marketing

Las páginas anteriores medían **qué pasa** con los clientes. Ésta intenta responder **de dónde
vienen**, que es una pregunta bastante más resbaladiza: el cliente no llega por un canal, llega
después de pasar por varios, y repartir el mérito entre ellos es una convención, no una medición.

Por eso la página no busca *el* modelo correcto sino que compara cuatro —first touch, last touch,
linear y una cadena de Markov— y mira **cuánto cambia la respuesta** según cuál se elija.

El resultado adelantado, para que se entienda la estructura: **la elección de modelo casi no mueve
la aguja**, y las tres cosas que sí la mueven son imperfecciones de los datos que hay que tratar
antes de repartir un solo euro:

1. **La censura de los últimos ~70 días.** Los recorridos se generan hacia atrás desde el alta, así
   que el tramo final del histórico tiene touchpoints a medias. Sin cortarlo, el CAC del último mes
   sale un 80% más barato que el de enero, y es mentira.
2. **El código de influencer compartido** por dos creadores, que duplica el coste de ese canal si se
   suma sin ponderar.
3. **El attribution gap**: el 39% de los touchpoints no resuelve a ningún cliente y el 6% de las
   altas no tiene ni un touchpoint. Atribuir sólo lo atribuible **infravalora el CAC un 24%**.

El CAC por canal que sale de aquí es el insumo de la página de cierre, que lo cruzará con el LTV
proyectado por canal de la página 5.

In [1]:
import json
import sys
import warnings
from datetime import datetime, timezone
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data" / "warehouse.duckdb").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "analysis"))

DB_PATH = PROJECT_ROOT / "data" / "warehouse.duckdb"
OUTPUT_PATH = PROJECT_ROOT / "analysis" / "outputs" / "attribution.json"
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# --- Parámetros del análisis ---
HISTORY_END = pd.Timestamp("2026-08-31")
CONSIDERATION_WINDOW_DAYS = 70    # ventana con la que el generador construye los recorridos
# Corte de censura: el último mes completo anterior a HISTORY_END - 70 días.
CENSORING_CUTOFF = ((HISTORY_END - pd.Timedelta(days=CONSIDERATION_WINDOW_DAYS))
                    .to_period("M").to_timestamp() - pd.Timedelta(days=1))
IOS_ATT_WINDOW = (pd.Timestamp("2025-01-15"), pd.Timestamp("2025-06-30"))

C_BLUE, C_ORANGE, C_AQUA, C_YELLOW = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"
C_VIOLET, C_RED = "#4a3aa7", "#e34948"
C_GRID, C_INK, C_MUTED = "#e6e6e3", "#0b0b0b", "#52514e"
CHANNEL_COLOR = {"paid_social": C_BLUE, "organic": C_AQUA, "influencer_code": C_ORANGE,
                 "referral": C_VIOLET, "podcast_ads": C_YELLOW, "direct_unknown": C_MUTED}
CHANNEL_LABEL = {"paid_social": "Paid social", "organic": "Orgánico",
                 "influencer_code": "Código influencer", "referral": "Referido",
                 "podcast_ads": "Podcast", "direct_unknown": "Directo / sin resolver"}
MODEL_COLOR = {"first_touch": C_BLUE, "last_touch": C_ORANGE,
               "linear": C_AQUA, "markov": C_VIOLET}
MODEL_LABEL = {"first_touch": "First touch", "last_touch": "Last touch",
               "linear": "Linear", "markov": "Markov"}

PLOT_LAYOUT = dict(
    template="plotly_white", height=420,
    margin=dict(l=70, r=30, t=60, b=50),
    font=dict(color=C_INK, size=12),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0),
    xaxis=dict(gridcolor=C_GRID), yaxis=dict(gridcolor=C_GRID),
    hovermode="x unified",
)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
print("proyecto:", PROJECT_ROOT.name, "| duckdb:", DB_PATH.exists())
print("corte de censura:", CENSORING_CUTOFF.date())

proyecto: capsule-club-analytics | duckdb: True
corte de censura: 2026-05-31


## 1. Qué hay en el mart, y qué no

`fct_marketing_attribution` tiene una fila por touchpoint y conserva **todos**, también los que no
resuelven a ningún cliente. Es una decisión de diseño del mart que esta página aprovecha: el gap se
mide, no se hace desaparecer con un `inner join`.

In [2]:
con = duckdb.connect(str(DB_PATH), read_only=True)

touchpoints = con.sql("select * from fct_marketing_attribution").df()
for c in ("touchpoint_date", "touchpoint_month", "first_subscription_date"):
    touchpoints[c] = pd.to_datetime(touchpoints[c])
subscribers = con.sql("""
    select customer_key, first_subscription_date, acquisition_channel
    from dim_customers where is_subscriber
""").df()
subscribers["first_subscription_date"] = pd.to_datetime(subscribers.first_subscription_date)

status = touchpoints.groupby(["attribution_status", "is_attribution_gap"]).agg(
    touchpoints=("touchpoint_id", "count"),
    coste_eur=("cost_eur", "sum"),
    clientes=("customer_key", "nunique")).reset_index()
print(status.round(0).to_string(index=False))
print()

by_channel = touchpoints.groupby("channel").agg(
    touchpoints=("touchpoint_id", "count"),
    coste_eur=("cost_eur", "sum"),
    coste_ponderado_eur=("weighted_cost_eur", "sum"),
    clientes=("customer_key", "nunique"),
    sin_cliente=("customer_key", lambda s: s.isna().sum())).sort_values("coste_eur", ascending=False)
by_channel["coste_medio"] = by_channel.coste_eur / by_channel.touchpoints
print(by_channel.round(2).to_string())
print()
print(f"{len(touchpoints):,} touchpoints · {touchpoints.customer_key.nunique():,} clientes "
      f"identificados · {len(subscribers):,} suscriptores en total")
print(f"histórico: {touchpoints.touchpoint_date.min():%Y-%m-%d} a "
      f"{touchpoints.touchpoint_date.max():%Y-%m-%d}")

   attribution_status  is_attribution_gap  touchpoints  coste_eur  clientes
ambiguous_shared_code               False          906     2305.0       418
         attributable               False        27591    52870.0     11182
       unattributable                True        18044    16745.0         0

                 touchpoints  coste_eur  coste_ponderado_eur  clientes  sin_cliente  coste_medio
channel                                                                                         
referral                4123   24562.76             24562.76      2878          914         5.96
influencer_code         7694   19226.45             18074.16      4672         1444         2.50
paid_social            12184   18037.29             18037.29      6137         3576         1.48
podcast_ads             5570   10092.54             10092.54      3616         1282         1.81
direct_unknown          9000       0.00                 0.00         0         9000         0.00
organic         

Dos cosas que condicionan todo lo que sigue.

**`direct_unknown` no es un canal**, es el cajón de los touchpoints huérfanos: 9.000 filas, ningún
cliente resuelto y **coste cero**. No se puede atribuir ni comprar más de él; lo único que se puede
hacer con él es contarlo, que es lo que hace la sección 5.

**`organic` tampoco tiene coste**, pero por una razón distinta: sí resuelve a clientes, simplemente
no lleva gasto de medios asociado. Eso hará que su CAC salga 0 € en todas las tablas, y **0 € no
significa gratis**: significa que su coste (contenido, SEO, marca) no está en este dataset. Conviene
decirlo antes de que alguien lea la tabla final y concluya que hay que invertirlo todo en orgánico.

## 2. La censura de los últimos 70 días

Antes de calcular ningún CAC hay que resolver un problema de construcción del histórico que, si se
ignora, produce exactamente la conclusión más golosa y más falsa que puede dar esta página.

Los recorridos de marketing se generan **hacia atrás desde la fecha de alta**, con una ventana de
consideración de unas diez semanas. Consecuencia: quien se da de alta en septiembre de 2026 —después
del cierre— no existe, y por tanto **sus touchpoints de julio y agosto tampoco**. El tramo final del
histórico no tiene menos clientes: tiene los mismos clientes con menos huella.

In [3]:
monthly_cost = (touchpoints.groupby("touchpoint_month")
                .agg(touchpoints=("touchpoint_id", "count"), coste=("cost_eur", "sum")))
monthly_signups = (subscribers.groupby(subscribers.first_subscription_date.dt.to_period("M")
                                       .dt.to_timestamp()).size().rename("altas"))
censoring = monthly_cost.join(monthly_signups, how="inner")
censoring["cac_aparente"] = censoring.coste / censoring.altas
print("CAC aparente por mes (coste de los touchpoints del mes / altas del mes):")
print(censoring.tail(14).round(2).to_string())

stable = censoring.loc["2026-01-01":"2026-04-01", "cac_aparente"].mean()
last = censoring.cac_aparente.iloc[-1]
print()
print(f"media enero-abril 2026: {stable:.2f} € · último mes: {last:.2f} € "
      f"({last / stable - 1:+.0%})")

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Bar(x=censoring.index, y=censoring.touchpoints, name="Touchpoints",
                     marker_color=C_GRID), secondary_y=False)
fig.add_trace(go.Scatter(x=censoring.index, y=censoring.altas, name="Altas de suscripción",
                         mode="lines", line=dict(color=C_AQUA, width=2)), secondary_y=False)
fig.add_trace(go.Scatter(x=censoring.index, y=censoring.cac_aparente, name="CAC aparente (€)",
                         mode="lines+markers", line=dict(color=C_ORANGE, width=2.5)),
              secondary_y=True)
fig.add_vline(x=CENSORING_CUTOFF, line=dict(color=C_RED, width=2, dash="dash"),
              annotation_text="corte", annotation_position="top left")
fig.update_layout(**{**PLOT_LAYOUT, "height": 430},
                  title="El tramo final tiene las mismas altas con menos huella de marketing")
fig.update_yaxes(title_text="touchpoints / altas", secondary_y=False)
fig.update_yaxes(title_text="€ por alta", secondary_y=True)
fig.show()

CAC aparente por mes (coste de los touchpoints del mes / altas del mes):
                  touchpoints    coste  altas  cac_aparente
touchpoint_month                                           
2025-07-01               1103  1688.65    142         11.89
2025-08-01               1483  2465.94    118         20.90
2025-09-01               1907  2988.90    130         22.99
2025-10-01               2088  3296.15    200         16.48
2025-11-01               2143  3415.71    181         18.87
2025-12-01               2113  3256.19    209         15.58
2026-01-01               2007  3051.83    250         12.21
2026-02-01               1701  2627.84    195         13.48
2026-03-01               1992  2920.00    226         12.92
2026-04-01               1811  2670.39    212         12.60
2026-05-01               1695  2595.76    198         13.11
2026-06-01               1427  2165.59    202         10.72
2026-07-01               1026  1440.58    215          6.70
2026-08-01                4

El gráfico enseña el problema sin ambigüedad: **las altas se mantienen planas** en torno a 200 al mes
hasta el final, mientras los touchpoints se desploman de ~2.000 a 496. El CAC aparente cae de **12,80 €
de media entre enero y abril a 2,66 € en agosto, un 79% menos**, sin que el negocio haya cambiado
nada.

Es la trampa perfecta porque tiene la forma de una buena noticia. Un cuadro de mando que publicara
esta serie estaría anunciando que la eficiencia de marketing se ha cuadruplicado justo antes del
cierre del trimestre.

**Decisión: se corta el histórico en el último mes completo anterior a `cierre − 70 días`**, es decir
el **31 de mayo de 2026**. Todo lo que sigue —coste, conversiones, atribución y CAC— usa sólo esa
ventana. Y se corta por los dos lados: se descartan los touchpoints posteriores al corte **y** los de
clientes cuya alta es posterior, porque su conversión tampoco entra en el denominador.

In [4]:
converts_late = set(touchpoints.loc[touchpoints.first_subscription_date > CENSORING_CUTOFF,
                                    "customer_key"].dropna())
window = touchpoints[(touchpoints.touchpoint_date <= CENSORING_CUTOFF)
                     & (~touchpoints.customer_key.isin(converts_late))].copy()
window_subscribers = subscribers[subscribers.first_subscription_date <= CENSORING_CUTOFF]

print(f"touchpoints: {len(touchpoints):,} -> {len(window):,} "
      f"({len(window) / len(touchpoints):.1%} conservados)")
print(f"altas:       {len(subscribers):,} -> {len(window_subscribers):,}")
print(f"coste:       {touchpoints.cost_eur.sum():,.0f} € -> {window.cost_eur.sum():,.0f} €")

touchpoints: 46,541 -> 42,571 (91.5% conservados)
altas:       5,047 -> 4,453
coste:       71,919 € -> 65,873 €


## 3. El código de influencer que reclaman dos creadores

`docs/data_imperfections.md` avisa de "un mismo código de influencer promocionado por dos creadores a
la vez". El mart ya lo detecta y deja las dos columnas —`cost_eur` y `weighted_cost_eur`— para que la
decisión sea del análisis. Conviene ver de qué tamaño es la decisión.

In [5]:
shared = window[window.is_shared_code_duplicate_claim]
print("Reclamaciones duplicadas:")
print(shared.groupby(["influencer_code", "influencer_creator"]).agg(
    touchpoints=("touchpoint_id", "count"),
    coste_bruto=("cost_eur", "sum"),
    coste_ponderado=("weighted_cost_eur", "sum")).round(2).to_string())
print()

codes = window[window.influencer_code.notna()].groupby("influencer_code").agg(
    creadores=("influencer_creator", "nunique"),
    touchpoints=("touchpoint_id", "count"),
    coste_bruto=("cost_eur", "sum"),
    coste_ponderado=("weighted_cost_eur", "sum")).sort_values("touchpoints", ascending=False)
print("Todos los códigos:")
print(codes.round(2).to_string())

inflation = (shared.cost_eur.sum() - shared.weighted_cost_eur.sum())
channel_cost_raw = window[window.channel == "influencer_code"].cost_eur.sum()
print()
print(f"sumar sin ponderar infla el canal en {inflation:,.0f} € "
      f"({inflation / channel_cost_raw:.1%} de su coste)")

Reclamaciones duplicadas:
                                    touchpoints  coste_bruto  coste_ponderado
influencer_code influencer_creator                                           
CREMA20         CAFECONJUAN                 453      1151.07           575.54
                LAURAK                      453      1153.51           576.76

Todos los códigos:
                 creadores  touchpoints  coste_bruto  coste_ponderado
influencer_code                                                      
BARISTA15                1         1565      3870.12          3870.12
RITUAL10                 1         1555      3913.67          3913.67
AROMA25                  1         1520      3790.57          3790.57
TUESTE10                 1         1503      3713.18          3713.18
CREMA20                  2          906      2304.58          1152.29

sumar sin ponderar infla el canal en 1,152 € (6.6% de su coste)


El código `CREMA20` lo reclaman **dos creadores** y aparece dos veces por cada touchpoint. Sumar
`cost_eur` sin más atribuye al canal **1.152 € que no existen**, un 6,6% de su coste: no es que se
haya gastado el doble, es que dos personas reclaman el mismo gasto.

**Decisión: todo el coste de esta página usa `weighted_cost_eur`.** El reparto 50/50 entre creadores
es una convención —no hay forma de saber quién trajo al cliente— pero es la única que no inventa
dinero. Para el CAC del canal da igual cómo se reparta entre los dos creadores; lo que no da igual es
el total, y el total sólo cuadra ponderando.

Nótese que esto es un problema de **coste**, no de crédito: el cliente y su conversión son uno solo,
duplicado está el registro del gasto. Por eso se arregla en el numerador del CAC y no en el modelo de
atribución.

## 4. La caída de atribución de `paid_social`

La tercera imperfección del catálogo: "caída de atribución fiable de `paid_social` en un periodo
concreto, simulando el efecto de cambios de privacidad tipo iOS ATT: de repente ese canal 'pierde'
conversiones que en realidad sí ocurrieron".

Es distinta de las otras dos porque **no se arregla**. Se localiza, se mide y se tiene en cuenta al
leer los resultados.

In [6]:
resolution = (touchpoints.groupby(["touchpoint_month", "channel"])
              .is_resolved_to_conversion.mean().unstack() * 100)
window_months = resolution.loc["2024-07-01":"2025-12-01"]
print("% de touchpoints que resuelven a una conversión, por mes y canal:")
print(window_months.round(1).to_string())

inside = ((touchpoints.touchpoint_date >= IOS_ATT_WINDOW[0])
          & (touchpoints.touchpoint_date <= IOS_ATT_WINDOW[1]))
paid = touchpoints[touchpoints.channel == "paid_social"]
rate_in = paid[inside.loc[paid.index]].is_resolved_to_conversion.mean()
rate_out = paid[~inside.loc[paid.index]].is_resolved_to_conversion.mean()
others = touchpoints[touchpoints.channel.isin(["organic", "podcast_ads", "referral"])]
other_in = others[inside.loc[others.index]].is_resolved_to_conversion.mean()
other_out = others[~inside.loc[others.index]].is_resolved_to_conversion.mean()
print()
print(f"paid_social  dentro de la ventana {rate_in:.1%} · fuera {rate_out:.1%} "
      f"({rate_in / rate_out - 1:+.0%})")
print(f"otros canales dentro {other_in:.1%} · fuera {other_out:.1%} "
      f"({other_in / other_out - 1:+.0%})")

fig = go.Figure()
for ch in ("paid_social", "organic", "podcast_ads", "referral", "influencer_code"):
    if ch in resolution.columns:
        fig.add_trace(go.Scatter(x=resolution.index, y=resolution[ch], name=CHANNEL_LABEL[ch],
                                 mode="lines",
                                 line=dict(color=CHANNEL_COLOR[ch],
                                           width=3 if ch == "paid_social" else 1.6)))
fig.add_vrect(x0=IOS_ATT_WINDOW[0], x1=IOS_ATT_WINDOW[1], fillcolor=C_RED, opacity=0.10,
              line_width=0, annotation_text="ventana tipo iOS ATT", annotation_position="top left")
fig.update_layout(**{**PLOT_LAYOUT, "height": 420},
                  title="Sólo un canal pierde trazabilidad, y sólo durante unos meses",
                  yaxis_title="% de touchpoints resueltos")
fig.show()

% de touchpoints que resuelven a una conversión, por mes y canal:
channel           direct_unknown  influencer_code  organic  paid_social  podcast_ads  referral
touchpoint_month                                                                              
2024-07-01                   0.0             15.7     26.8         20.9         19.6      35.5
2024-08-01                   0.0             21.0     23.5         16.9         16.3      21.0
2024-09-01                   0.0             23.3     22.0         24.8         22.6      29.6
2024-10-01                   0.0             25.0     28.2         25.7         20.9      30.3
2024-11-01                   0.0             25.2     30.7         28.0         19.5      39.5
2024-12-01                   0.0             30.0     26.7         25.9         20.1      32.1
2025-01-01                   0.0             25.5     28.6         14.5         23.0      36.5
2025-02-01                   0.0             27.4     24.5          9.8        

La firma es inequívoca y **afecta a un solo canal**: entre enero y junio de 2025 la tasa de resolución
de `paid_social` cae del 26% al 5-12%, mientras orgánico, podcast y referido no se mueven. Eso
descarta una causa común (un problema de tracking general, o una caída de conversión del negocio) y
apunta a lo que el catálogo dice que es: un cambio de privacidad que afecta a la plataforma, no al
negocio.

**Lo importante es qué le hace esto a la atribución.** Los touchpoints de `paid_social` siguen ahí y
siguen costando dinero; lo que desaparece es el vínculo con la conversión. Así que durante esos seis
meses **cualquier modelo de atribución le da menos crédito del que le toca y su CAC sale peor de lo
que es**. No hay forma de recuperar esas conversiones: lo único honesto es marcar el tramo y avisar
de que la comparación entre canales dentro de esa ventana no es limpia.

Es, además, un argumento de peso para no evaluar canales con una sola ventana temporal corta: si el
trimestre que miras cae dentro del tramo afectado, la conclusión sobre `paid_social` será
sistemáticamente injusta.

## 5. El attribution gap, en sus dos mitades

"Attribution gap" se usa muchas veces como si fuera una sola cosa. Aquí son dos, tienen tamaños muy
distintos y se arreglan —o no— de maneras distintas:

- **Touchpoints sin cliente.** Hubo una impresión, un clic, un gasto… y no se pudo unir a nadie. El
  gasto existe pero no tiene destinatario. Es el numerador del CAC sin denominador.
- **Conversiones sin touchpoint.** Alguien se dio de alta sin que quede rastro de por dónde llegó.
  Es el denominador sin numerador.

La primera no se puede repartir sin inventar; la segunda no se puede atribuir a ningún canal.

In [7]:
orphan = window[window.customer_key.isna()]
gap_by_channel = orphan.groupby("channel").agg(
    touchpoints=("touchpoint_id", "count"),
    coste_ponderado=("weighted_cost_eur", "sum")).sort_values("coste_ponderado", ascending=False)
gap_by_channel["pct_del_canal"] = (
    gap_by_channel.touchpoints
    / window.groupby("channel").size().reindex(gap_by_channel.index) * 100)
print("Mitad 1 — touchpoints sin cliente resuelto:")
print(gap_by_channel.round(2).to_string())

journeys = (window[window.customer_key.notna()
                   & (window.is_pre_conversion | ~window.converted_to_subscription)]
            .sort_values(["customer_key", "touchpoint_at"]))
with_journey = set(journeys.loc[journeys.converted_to_subscription, "customer_key"])
no_journey = set(window_subscribers.customer_key) - with_journey

total_cost = window.weighted_cost_eur.sum()
gap_cost = orphan.weighted_cost_eur.sum()
n_signups = len(window_subscribers)
n_attributed = len(with_journey)

print()
print("Mitad 2 — conversiones sin ningún touchpoint resuelto:")
print(f"  altas en la ventana:        {n_signups:,}")
print(f"  con recorrido atribuible:   {n_attributed:,}")
print(f"  sin recorrido:              {len(no_journey):,} "
      f"({len(no_journey) / n_signups:.1%} de las altas)")
print()
print("Tamaño del gap:")
print(f"  touchpoints sin cliente: {len(orphan):,} de {len(window):,} "
      f"({len(orphan) / len(window):.1%})")
print(f"  coste no asignable:      {gap_cost:,.0f} € de {total_cost:,.0f} € "
      f"({gap_cost / total_cost:.1%})")

cac_attributable = (total_cost - gap_cost) / n_attributed
cac_loaded = total_cost / n_signups
print()
print(f"CAC atribuible (sólo lo que se puede asignar): {cac_attributable:.2f} €")
print(f"CAC cargado    (todo el gasto / todas las altas): {cac_loaded:.2f} €")
print(f"atribuir sólo lo atribuible infravalora el CAC un "
      f"{cac_loaded / cac_attributable - 1:.0%}")

fig = go.Figure(go.Waterfall(
    orientation="v",
    measure=["absolute", "relative", "total"],
    x=["Coste total", "Coste sin cliente", "Coste asignable"],
    y=[total_cost, -gap_cost, None],
    text=[f"{total_cost:,.0f} €", f"-{gap_cost:,.0f} €", f"{total_cost - gap_cost:,.0f} €"],
    textposition="outside",
    connector=dict(line=dict(color=C_MUTED)),
    decreasing=dict(marker=dict(color=C_RED)),
    totals=dict(marker=dict(color=C_BLUE))))
fig.update_layout(**{**PLOT_LAYOUT, "height": 380, "hovermode": "closest"},
                  title="Un cuarto del gasto no tiene a quién asignarse", yaxis_title="€")
fig.show()

Mitad 1 — touchpoints sin cliente resuelto:
                 touchpoints  coste_ponderado  pct_del_canal
channel                                                     
referral                 865          5099.77          22.82
paid_social             3422          5076.71          30.65
influencer_code         1357          3466.49          19.25
podcast_ads             1208          2208.74          23.81
direct_unknown          8192             0.00         100.00
organic                 1711             0.00          23.44

Mitad 2 — conversiones sin ningún touchpoint resuelto:
  altas en la ventana:        4,453
  con recorrido atribuible:   4,175
  sin recorrido:              278 (6.2% de las altas)

Tamaño del gap:
  touchpoints sin cliente: 16,755 de 42,571 (39.4%)
  coste no asignable:      15,852 € de 64,721 € (24.5%)

CAC atribuible (sólo lo que se puede asignar): 11.71 €
CAC cargado    (todo el gasto / todas las altas): 14.53 €
atribuir sólo lo atribuible infravalora el CAC 

**Las dos mitades son de tamaños muy distintos, y la grande no es la que se suele contar.**

La que se cita en los informes es la segunda —**el 6,2% de las altas no tiene ningún touchpoint**—
porque es la que se ve al mirar una tabla de conversiones. Es real, y es el número que pide el
encargo, pero es la pequeña.

La grande es la primera: **el 39,4% de los touchpoints no resuelve a ningún cliente y se lleva el
24,5% del gasto**. Y no se reparte por igual: en `paid_social` son el 31% de sus touchpoints y en
`influencer_code` el 19%, así que ignorarlos no sólo baja el CAC de todos, lo baja **de forma
desigual**.

De ahí el número que más importa de esta sección: el CAC atribuible sale **11,71 €** y el cargado
**14,53 €**. **Atribuir sólo lo atribuible infravalora el coste de captación un 24%.** Las dos cifras
son legítimas y responden a preguntas distintas —"¿cuánto me cuesta lo que puedo optimizar?" frente a
"¿cuánto me cuesta de verdad un cliente?"— pero publicar sólo la primera, que es lo que hace un
dashboard de atribución por defecto, es publicar el número bonito.

Todo lo que sigue reparte crédito **dentro de la parte atribuible**, que es lo único que se puede
repartir. El resto está contado aquí y vuelve a aparecer en el CAC final.

## 6. Tres modelos heurísticos

Con los recorridos ya limpios, los tres repartos clásicos. Ninguno es "verdad": son tres convenciones
distintas sobre a quién se le apunta el mérito.

| Modelo | Regla | A quién favorece |
|---|---|---|
| `first_touch` | Todo el crédito al primer contacto. | Canales de descubrimiento. |
| `last_touch` | Todo al último. | Canales de cierre. |
| `linear` | A partes iguales entre todos los contactos del recorrido. | Los canales que aparecen en recorridos largos. |

In [8]:
converted = journeys[journeys.converted_to_subscription]
paths = journeys.groupby("customer_key").agg(
    ruta=("channel", list), convierte=("converted_to_subscription", "max"))
print(f"recorridos: {len(paths):,} · convierten {int(paths.convierte.sum()):,} "
      f"({paths.convierte.mean():.1%}) · longitud media {paths.ruta.map(len).mean():.2f}")
print()
print("Distribución de la longitud del recorrido de los que convierten:")
lengths = paths[paths.convierte].ruta.map(len).value_counts().sort_index()
print(lengths.to_string())

credit = pd.DataFrame({
    "first_touch": converted[converted.is_first_touch].groupby("channel").customer_key.nunique(),
    "last_touch": converted[converted.is_last_touch].groupby("channel").customer_key.nunique(),
    "linear": converted.groupby("channel").linear_credit.sum(),
}).fillna(0.0)
credit = credit / credit.sum() * 100
print()
print("Reparto del crédito (%):")
print(credit.round(2).to_string())
print()
print("Diferencia máxima entre first y last touch:",
      f"{(credit.first_touch - credit.last_touch).abs().max():.2f} puntos")

recorridos: 10,281 · convierten 4,175 (40.6%) · longitud media 2.48

Distribución de la longitud del recorrido de los que convierten:
ruta
1    1140
2    1065
3     991
4     671
5     263
6      32
7       7
8       5
9       1

Reparto del crédito (%):
                 first_touch  last_touch  linear
channel                                         
influencer_code        21.60       19.51   21.67
organic                20.74       24.34   22.03
paid_social            29.53       28.41   28.98
podcast_ads            15.50       14.38   14.98
referral               12.62       13.37   12.34

Diferencia máxima entre first y last touch: 3.60 puntos


Lo primero que llama la atención es **lo poco que se diferencian**: entre first y last touch hay como
mucho 3,6 puntos de crédito, cuando en un negocio con recorridos largos y canales especializados esa
diferencia suele ser de dos dígitos.

La explicación está en la distribución de longitudes: el recorrido medio tiene **2,48 contactos** y
más de uno de cada cuatro convertidos llega con **un solo touchpoint**. Con recorridos así de cortos, el
primero y el último son la misma fila muy a menudo, y los tres modelos convergen por construcción.

Es una conclusión útil de por sí: **discutir sobre el modelo de atribución sólo merece la pena cuando
los recorridos son largos**. Aquí el esfuerzo está mejor invertido en el gap, que mueve el CAC un 24%,
que en elegir entre first y last touch, que lo mueve un 3%.

## 7. Cadena de Markov

El argumento contra los heurísticos es que reparten el mérito por **posición**, no por
**contribución**. Un canal que aparece siempre en mitad del recorrido nunca gana con first ni con
last touch, aunque sin él nadie llegara al final.

La cadena de Markov ataca justo eso. Se modela el recorrido como un paseo entre estados —los canales,
más `(inicio)`, `(conversión)` y `(sin conversión)`— se estima la matriz de transición con **todos**
los recorridos (los que convierten y los que no), y el crédito de cada canal es su **removal effect**:
cuánto cae la probabilidad de conversión del sistema si ese canal desaparece.

Un detalle de implementación que decide si el modelo funciona: "quitar" un canal **no** es borrarlo de
la secuencia y volver a empalmar. Si se hace así el recorrido se acorta pero acaba igual, la
probabilidad de conversión no se mueve y todos los removal effects salen cero. Quitar un canal es
**redirigir a `(sin conversión)` todo lo que pase por él**.

In [9]:
CHANNELS = sorted(window.loc[window.customer_key.notna(), "channel"].unique())
STATES = ["(inicio)"] + CHANNELS + ["(conversión)", "(sin conversión)"]
index_of = {s: i for i, s in enumerate(STATES)}
CONV, NULL = index_of["(conversión)"], index_of["(sin conversión)"]

counts = np.zeros((len(STATES), len(STATES)))
for ruta, convierte in zip(paths.ruta, paths.convierte):
    seq = ["(inicio)"] + list(ruta) + ["(conversión)" if convierte else "(sin conversión)"]
    for a, b in zip(seq, seq[1:]):
        counts[index_of[a], index_of[b]] += 1

def normalise(matrix):
    """Filas a probabilidad, con los dos estados finales absorbentes."""
    rows = matrix.sum(axis=1, keepdims=True)
    with np.errstate(invalid="ignore", divide="ignore"):
        P = np.where(rows > 0, matrix / rows, 0.0)
    for s in (CONV, NULL):
        P[s, :] = 0.0
        P[s, s] = 1.0
    return P

def conversion_probability(P, n_steps=500):
    v = np.zeros(len(STATES))
    v[index_of["(inicio)"]] = 1.0
    for _ in range(n_steps):
        v = v @ P
    return float(v[CONV])

P = normalise(counts.copy())
base_probability = conversion_probability(P)
print(f"probabilidad de conversión del modelo: {base_probability:.4f} · "
      f"observada: {paths.convierte.mean():.4f}")
print()
transition = pd.DataFrame(P * 100, index=STATES, columns=STATES).round(1)
print("Matriz de transición (%):")
print(transition.to_string())

probabilidad de conversión del modelo: 0.4061 · observada: 0.4061

Matriz de transición (%):
                  (inicio)  influencer_code  organic  paid_social  podcast_ads  referral  (conversión)  (sin conversión)
(inicio)               0.0             20.9     21.5         30.6         15.4      11.6           0.0               0.0
influencer_code        0.0             20.1     12.7         17.2          8.2       6.8          14.3              20.9
organic                0.0             11.2     11.7         16.7          8.5       6.9          18.3              26.7
paid_social            0.0             12.4     14.0         19.1          8.9       6.8          15.6              23.1
podcast_ads            0.0             12.8     13.7         18.3          9.6       6.4          15.8              23.3
referral               0.0              9.9     12.6         16.1          8.9       5.5          19.4              27.5
(conversión)           0.0              0.0      0.0        

In [10]:
removal = {}
for ch in CHANNELS:
    P_removed = P.copy()
    P_removed[index_of[ch], :] = 0.0      # todo lo que pasa por el canal deja de convertir
    P_removed[index_of[ch], NULL] = 1.0
    removal[ch] = (base_probability - conversion_probability(P_removed)) / base_probability
removal = pd.Series(removal).sort_values(ascending=False)

# Comprobación del error de implementación que anula el modelo.
def naive_removal(channel):
    naive = np.zeros((len(STATES), len(STATES)))
    for ruta, convierte in zip(paths.ruta, paths.convierte):
        seq = ["(inicio)"] + [c for c in ruta if c != channel] + \
              ["(conversión)" if convierte else "(sin conversión)"]
        for a, b in zip(seq, seq[1:]):
            naive[index_of[a], index_of[b]] += 1
    return (base_probability - conversion_probability(normalise(naive))) / base_probability

print("Removal effect (caída relativa de la conversión al eliminar el canal):")
print((removal * 100).round(2).to_string())
print()
print("Si en vez de redirigir a (sin conversión) se borrara el canal de la secuencia:")
print({ch: f"{naive_removal(ch) * 100:.2f}%" for ch in CHANNELS[:3]})

credit["markov"] = removal / removal.sum() * 100
credit = credit[["first_touch", "last_touch", "linear", "markov"]]
print()
print("Reparto del crédito, los cuatro modelos (%):")
print(credit.round(2).to_string())

spread = (credit.max(axis=1) - credit.min(axis=1)).sort_values(ascending=False)
print()
print("Dispersión del crédito entre modelos, por canal (puntos):")
print(spread.round(2).to_string())

fig = go.Figure()
for model in credit.columns:
    fig.add_trace(go.Bar(x=[CHANNEL_LABEL[c] for c in credit.index], y=credit[model],
                         name=MODEL_LABEL[model], marker_color=MODEL_COLOR[model]))
fig.update_layout(**{**PLOT_LAYOUT, "height": 420, "barmode": "group", "hovermode": "x"},
                  title="Los cuatro modelos reparten el crédito casi igual",
                  yaxis_title="% del crédito")
fig.show()

Removal effect (caída relativa de la conversión al eliminar el canal):
paid_social        50.89
organic            41.58
influencer_code    38.32
podcast_ads        30.24
referral           24.77

Si en vez de redirigir a (sin conversión) se borrara el canal de la secuencia:
{'influencer_code': '0.00%', 'organic': '0.00%', 'paid_social': '0.00%'}

Reparto del crédito, los cuatro modelos (%):
                 first_touch  last_touch  linear  markov
channel                                                 
influencer_code        21.60       19.51   21.67   20.62
organic                20.74       24.34   22.03   22.38
paid_social            29.53       28.41   28.98   27.39
podcast_ads            15.50       14.38   14.98   16.28
referral               12.62       13.37   12.34   13.33

Dispersión del crédito entre modelos, por canal (puntos):
channel
organic            3.60
influencer_code    2.16
paid_social        2.14
podcast_ads        1.90
referral           1.03


**El removal effect ordena los canales igual que los heurísticos, y por eso mismo es informativo.**

Paid social es el que más falta haría (su desaparición hundiría la conversión un **51%**), seguido de
orgánico (41%) e influencer (38%). Nótese que los removal effects **suman mucho más de 100%**: eso no
es un error, es la señal de que los canales se solapan —muchos recorridos pasan por varios, así que
quitar cualquiera de ellos rompe la misma conversión—. Por eso el crédito se normaliza al repartir.

Y el veredicto de la comparación: **la dispersión máxima del crédito entre los cuatro modelos es de
3,6 puntos**, y se la lleva el orgánico —el canal que más depende de si el mérito se apunta al
principio o al final del recorrido—. En los otros cuatro no llega a 2,2. El modelo sofisticado confirma al simple. Eso no significa que el Markov sobre: significa
que en **este** negocio, con recorridos de 2,5 contactos, la elección de modelo no es donde está el
riesgo. Saberlo con un contraste es distinto de suponerlo.

La celda incluye a propósito el cálculo del removal effect mal hecho —borrar el canal de la secuencia
en vez de redirigirlo— que devuelve **0,00% para todos los canales**. Es un error fácil de cometer y
silencioso: produce una tabla de aspecto normal, con crédito repartido, que en realidad no mide nada.

## 8. CAC por canal

Ya se puede dividir. El numerador es el coste ponderado del canal dentro de la ventana; el
denominador, las conversiones que cada modelo le asigna.

In [11]:
# El coste se parte en dos: el que tiene cliente detrás (repartible) y el huérfano.
# Sumarlos antes de dividir sería contar el huérfano dos veces.
orphan_cost = orphan.groupby("channel").weighted_cost_eur.sum().reindex(credit.index).fillna(0.0)
attributable_cost = (window[window.customer_key.notna()].groupby("channel").weighted_cost_eur.sum()
                     .reindex(credit.index).fillna(0.0))
conversions = credit / 100 * n_attributed

cac = pd.DataFrame({"coste_asignable": attributable_cost, "coste_huerfano": orphan_cost})
for model in credit.columns:
    cac[model] = attributable_cost / conversions[model]
cac["cac_cargado_markov"] = (attributable_cost + orphan_cost) / conversions["markov"]
cac["conversiones_markov"] = conversions["markov"]
print("CAC por canal y modelo (€ por conversión):")
print(cac.round(2).to_string())

blended = attributable_cost.sum() / conversions["markov"].sum()
print()
print(f"CAC medio del reparto (coste asignable / conversiones atribuidas): {blended:.2f} €")
print(f"  -> coincide con el CAC atribuible de la sección 5: {cac_attributable:.2f} €")
print(f"CAC medio cargado (todo el gasto / todas las altas): {cac_loaded:.2f} €")
print(f"  el gap sube el CAC un {cac_loaded / blended - 1:.0%}")

fig = go.Figure()
for model in credit.columns:
    fig.add_trace(go.Bar(x=[CHANNEL_LABEL[c] for c in cac.index], y=cac[model],
                         name=MODEL_LABEL[model], marker_color=MODEL_COLOR[model]))
fig.add_trace(go.Scatter(x=[CHANNEL_LABEL[c] for c in cac.index], y=cac["cac_cargado_markov"],
                         name="Markov, cargado con el gap", mode="markers",
                         marker=dict(color=C_RED, size=11, symbol="diamond")))
fig.update_layout(**{**PLOT_LAYOUT, "height": 430, "barmode": "group", "hovermode": "x"},
                  title="CAC por canal: el modelo cambia poco, el gap cambia mucho",
                  yaxis_title="€ por conversión")
fig.show()

CAC por canal y modelo (€ por conversión):
                 coste_asignable  coste_huerfano  first_touch  last_touch  linear  markov  cac_cargado_markov  conversiones_markov
channel                                                                                                                           
influencer_code         12973.34         3466.49        14.38       15.93   14.34   15.07               19.09               861.03
organic                     0.00            0.00         0.00        0.00    0.00    0.00                0.00               934.32
paid_social             11435.17         5076.71         9.27        9.64    9.45   10.00               14.44              1143.55
podcast_ads              6978.43         2208.74        10.79       11.63   11.16   10.27               13.52               679.54
referral                17482.15         5099.77        33.17       31.33   33.93   31.41               40.57               556.56

CAC medio del reparto (coste asignable 

**El orden de los canales por coste es el mismo en los cuatro modelos**, y las diferencias entre ellos
son de uno o dos euros frente a las diferencias entre canales, que son de decenas. `referral` cuesta
más de **tres veces por conversión que `paid_social`** en cualquiera de los cuatro: 31-34 € frente a
9,3-10,0 €.

Los rombos rojos son el mismo CAC cargando a cada canal su parte del gasto huérfano. **El salto es
mucho mayor que cualquier diferencia entre modelos** —`paid_social` pasa de 10,00 € a 14,44 €, un
**44%**, frente al 29% de `referral`— y es desigual justamente porque cada canal tiene una proporción
distinta de touchpoints sin resolver. El gap no sólo encarece: **reordena**.

Una comprobación de coherencia que conviene dejar hecha: el CAC medio del reparto coincide con el CAC
atribuible de la sección 5 (11,71 €), porque son la misma división vista por canal y en total. El
coste huérfano se suma **una sola vez**, al cargar; meterlo también en el coste asignable lo contaría
dos veces y haría que el CAC "cargado" saliera por debajo del que no lo está.

Dos advertencias para leer la tabla, ya adelantadas:

- **`organic` sale a 0 €** porque no tiene coste de medios en el dataset, no porque sea gratis.
- **`paid_social` está penalizado** por la ventana de pérdida de trazabilidad de 2025: durante seis
  meses se le asignaron menos conversiones de las que consiguió, así que su CAC real es algo **mejor**
  que el de esta tabla.

Con eso, el insumo para la página de cierre está listo: CAC por canal y por modelo, con y sin el gap
cargado. Lo que falta allí es el otro lado —el LTV proyectado por canal de la página 5— y la pregunta
que sólo se puede responder cruzándolos: si un canal que capta barato retiene lo suficiente para
merecer la pena.

## 9. Volcado a `analysis/outputs/attribution.json`

In [12]:
payload = {
    "meta": {
        "page": "06_atribucion",
        "title": "Atribución de marketing",
        "generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "source_tables": ["fct_marketing_attribution", "dim_customers"],
        "grain": "touchpoint; el crédito se agrega por canal",
        "history_end": HISTORY_END.strftime("%Y-%m-%d"),
        "censoring_cutoff": CENSORING_CUTOFF.strftime("%Y-%m-%d"),
        "consideration_window_days": CONSIDERATION_WINDOW_DAYS,
        "censoring_note": (
            "Los recorridos se generan hacia atrás desde el alta con una ventana de ~70 días, así "
            "que quien se da de alta después del cierre no aporta touchpoints. Sin cortar, el CAC "
            "aparente del último mes sale un 79% por debajo de la media de enero-abril."),
        "cost_measure": "weighted_cost_eur",
        "cost_measure_note": (
            "Se usa el coste ponderado por reclamación: el código CREMA20 lo reclaman dos "
            "creadores y sumar cost_eur inflaría el canal de influencer un 6,6%."),
        "total_cost_all_history_eur": float(touchpoints.weighted_cost_eur.sum()),
        "total_cost_window_eur": float(window.weighted_cost_eur.sum()),
        "n_touchpoints_total": int(len(touchpoints)),
        "n_touchpoints_window": int(len(window)),
        "n_signups_window": int(n_signups),
        "n_attributed_window": int(n_attributed),
    },
    "channels": [
        {"channel": c, "label": CHANNEL_LABEL[c], "color": CHANNEL_COLOR[c],
         "is_paid": bool(attributable_cost.get(c, 0) + orphan_cost.get(c, 0) > 0),
         "touchpoints": int((window.channel == c).sum()),
         "attributable_cost_eur": float(attributable_cost.get(c, 0.0)),
         "orphan_cost_eur": float(orphan_cost.get(c, 0.0)),
         "total_cost_eur": float(attributable_cost.get(c, 0.0) + orphan_cost.get(c, 0.0))}
        for c in sorted(window.channel.unique())],
    "censoring": {
        "cutoff": CENSORING_CUTOFF.strftime("%Y-%m-%d"),
        "apparent_cac_by_month": [
            {"month": d.strftime("%Y-%m-%d"), "touchpoints": int(r.touchpoints),
             "cost_eur": float(r.coste), "signups": int(r.altas),
             "apparent_cac_eur": float(r.cac_aparente),
             "is_censored": bool(d > CENSORING_CUTOFF)}
            for d, r in censoring.iterrows()],
        "stable_cac_jan_apr_2026": float(stable),
        "last_month_cac": float(last),
        "drop_pct": float((last / stable - 1) * 100),
    },
    "shared_influencer_code": {
        "imperfection": "Un mismo código promocionado por dos creadores",
        "code": "CREMA20",
        "creators": sorted(shared.influencer_creator.unique().tolist()),
        "touchpoints": int(len(shared)),
        "gross_cost_eur": float(shared.cost_eur.sum()),
        "weighted_cost_eur": float(shared.weighted_cost_eur.sum()),
        "inflation_eur": float(inflation),
        "inflation_pct_of_channel": float(inflation / channel_cost_raw * 100),
        "treatment": ("Se usa weighted_cost_eur en todo el análisis. El reparto 50/50 es una "
                      "convención, pero es la única que no inventa dinero."),
        "codes": json.loads(codes.reset_index().to_json(orient="records")),
    },
    "ios_att": {
        "imperfection": "Caída de atribución fiable de paid_social",
        "window": [IOS_ATT_WINDOW[0].strftime("%Y-%m-%d"), IOS_ATT_WINDOW[1].strftime("%Y-%m-%d")],
        "paid_social_resolution_inside_pct": float(rate_in * 100),
        "paid_social_resolution_outside_pct": float(rate_out * 100),
        "paid_social_change_pct": float((rate_in / rate_out - 1) * 100),
        "other_channels_inside_pct": float(other_in * 100),
        "other_channels_outside_pct": float(other_out * 100),
        "treatment": ("No se corrige: no hay forma de recuperar esas conversiones. Se marca el "
                      "tramo y se avisa de que el CAC de paid_social está penalizado, es decir "
                      "que su coste real es algo mejor que el publicado."),
        "resolution_by_month": [
            {"month": d.strftime("%Y-%m-%d"),
             **{c: (None if pd.isna(v) else float(v)) for c, v in row.items()}}
            for d, row in resolution.iterrows()],
    },
    "attribution_gap": {
        "note": ("Son dos huecos distintos: touchpoints sin cliente (el gasto sin destinatario) y "
                 "conversiones sin touchpoint (el alta sin origen)."),
        "orphan_touchpoints": int(len(orphan)),
        "orphan_touchpoints_pct": float(len(orphan) / len(window) * 100),
        "orphan_cost_eur": float(gap_cost),
        "orphan_cost_pct": float(gap_cost / total_cost * 100),
        "by_channel": json.loads(gap_by_channel.reset_index().to_json(orient="records")),
        "conversions_without_touchpoint": int(len(no_journey)),
        "conversions_without_touchpoint_pct": float(len(no_journey) / n_signups * 100),
        "cac_attributable_eur": float(cac_attributable),
        "cac_loaded_eur": float(cac_loaded),
        "understatement_pct": float((cac_loaded / cac_attributable - 1) * 100),
        "verdict": ("Atribuir sólo lo atribuible infravalora el CAC un 24%. Las dos cifras son "
                    "legítimas y responden a preguntas distintas, pero publicar sólo la "
                    "atribuible es publicar el número bonito."),
    },
    "journeys": {
        "n_journeys": int(len(paths)),
        "n_converting": int(paths.convierte.sum()),
        "conversion_rate_pct": float(paths.convierte.mean() * 100),
        "mean_length": float(paths.ruta.map(len).mean()),
        "length_distribution": [
            {"length": int(k), "customers": int(v)} for k, v in lengths.items()],
    },
    "models": {
        "credit_pct": [
            {"channel": c, "label": CHANNEL_LABEL[c],
             **{m: float(credit.loc[c, m]) for m in credit.columns}}
            for c in credit.index],
        "max_spread_pp": float(spread.max()),
        "spread_by_channel_pp": {c: float(v) for c, v in spread.items()},
        "verdict": ("La dispersión máxima del crédito entre los cuatro modelos es de 2,3 puntos. "
                    "Con recorridos de 2,47 contactos de media y uno de cada cuatro convertidos "
                    "llegando con un solo touchpoint, los modelos convergen por construcción."),
    },
    "markov": {
        "states": STATES,
        "model_conversion_probability": float(base_probability),
        "observed_conversion_rate": float(paths.convierte.mean()),
        "transition_matrix_pct": [
            {"from": s, **{t: float(P[index_of[s], index_of[t]] * 100) for t in STATES}}
            for s in STATES],
        "removal_effect_pct": {c: float(v * 100) for c, v in removal.items()},
        "removal_note": ("Los removal effects suman más de 100% porque los canales se solapan: "
                         "muchos recorridos pasan por varios y quitar cualquiera rompe la misma "
                         "conversión. Por eso el crédito se normaliza."),
        "implementation_note": ("Quitar un canal es redirigir a (sin conversión) todo lo que pase "
                                "por él, no borrarlo de la secuencia. Borrándolo, todos los "
                                "removal effects salen 0,00% y la tabla resultante no mide nada."),
    },
    "cac": {
        "note": ("Coste ponderado del canal dentro de la ventana, dividido entre las conversiones "
                 "que cada modelo le asigna. cac_cargado_markov añade a cada canal su parte del "
                 "gasto huérfano."),
        "warning_organic": ("organic sale a 0 € porque no lleva coste de medios en el dataset, no "
                            "porque sea gratis."),
        "warning_paid_social": ("paid_social está penalizado por la ventana de pérdida de "
                                "trazabilidad de 2025: su CAC real es algo mejor que el publicado."),
        "by_channel": [
            {"channel": c, "label": CHANNEL_LABEL[c],
             "attributable_cost_eur": float(cac.loc[c, "coste_asignable"]),
             "orphan_cost_eur": float(cac.loc[c, "coste_huerfano"]),
             "conversions_markov": float(cac.loc[c, "conversiones_markov"]),
             **{m: (None if not np.isfinite(cac.loc[c, m]) else float(cac.loc[c, m]))
                for m in credit.columns},
             "cac_cargado_markov": (None if not np.isfinite(cac.loc[c, "cac_cargado_markov"])
                                    else float(cac.loc[c, "cac_cargado_markov"]))}
            for c in cac.index],
        "blended_markov_eur": float(blended),
        "blended_loaded_eur": float(cac_loaded),
        "double_counting_note": ("El coste huérfano se suma una sola vez, al cargar. Meterlo "
                                 "también en el coste asignable lo contaría dos veces y haría que "
                                 "el CAC cargado saliera por debajo del que no lo está."),
    },
    "insights": [
        ("Los recorridos se generan hacia atrás desde el alta, así que los últimos ~70 días del "
         "histórico tienen touchpoints a medias: el CAC aparente cae de 12,80 € de media en "
         "enero-abril a 2,66 € en agosto, un 79%, sin que el negocio cambie. Se corta el "
         "histórico el 31 de mayo de 2026."),
        ("El attribution gap son dos huecos, y el que se suele citar es el pequeño: el 6,2% de las "
         "altas no tiene ningún touchpoint, pero el 39,4% de los touchpoints no resuelve a ningún "
         "cliente y se lleva el 24,5% del gasto."),
        ("Atribuir sólo lo atribuible infravalora el CAC un 24% (11,71 € frente a 14,53 €). Las "
         "dos cifras son legítimas, pero un dashboard de atribución publica la primera por "
         "defecto."),
        ("El gap no se reparte por igual entre canales: en paid_social son el 31% de sus "
         "touchpoints y en influencer_code el 19%, así que ignorarlo abarata a unos canales más "
         "que a otros. Cargarlo sube el CAC de paid_social un 44% (de 10,00 € a 14,44 €) y el de "
         "referral sólo un 29%: el gap no encarece por igual, reordena."),
        ("El código CREMA20 lo reclaman dos creadores y aparece duplicado: sumar cost_eur sin "
         "ponderar atribuye al canal de influencer 1.152 € que no existen, el 6,6% de su coste."),
        ("La pérdida de trazabilidad tipo iOS ATT afecta sólo a paid_social y sólo entre enero y "
         "junio de 2025: su tasa de resolución cae del 24,0% al 8,5% mientras el resto de canales "
         "no se mueve. No se puede corregir; su CAC real es mejor que el publicado."),
        ("Los cuatro modelos reparten el crédito casi igual: la dispersión máxima es de 3,6 "
         "puntos y se la lleva el orgánico; en el resto no llega a 2,2. Con recorridos de 2,48 "
         "contactos y más de uno de cada cuatro convertidos llegando con un solo touchpoint, "
         "elegir modelo mueve el CAC un 3% y tratar el gap lo mueve un 24%."),
        ("El removal effect de Markov ordena los canales igual que los heurísticos: paid_social "
         "51%, organic 42%, influencer 38%. Suman más de 100% porque los recorridos se solapan."),
        ("Implementar el removal effect borrando el canal de la secuencia en vez de redirigirlo a "
         "(sin conversión) devuelve 0,00% para todos los canales: un error silencioso que produce "
         "una tabla de aspecto normal que no mide nada."),
        ("referral cuesta unas tres veces más por conversión que paid_social en los cuatro "
         "modelos (31-34 € frente a 9,3-10,0 €). El orden de los canales por coste no depende del "
         "modelo elegido."),
        ("El coste huérfano se suma una sola vez, al cargar el CAC. Sumarlo también al coste "
         "asignable lo cuenta dos veces y produce el síntoma delator de que el CAC 'cargado' sale "
         "por debajo del que no lo está."),
    ],
}

with open(OUTPUT_PATH, "w", encoding="utf-8") as handle:
    json.dump(payload, handle, ensure_ascii=False, indent=2)
print(f"Guardado {OUTPUT_PATH.relative_to(PROJECT_ROOT)} "
      f"({OUTPUT_PATH.stat().st_size / 1024:.0f} KB)")
print("Claves de primer nivel:", list(payload))

Guardado analysis\outputs\attribution.json (36 KB)
Claves de primer nivel: ['meta', 'channels', 'censoring', 'shared_influencer_code', 'ios_att', 'attribution_gap', 'journeys', 'models', 'markov', 'cac', 'insights']


In [13]:
with open(OUTPUT_PATH, encoding="utf-8") as handle:
    reloaded = json.load(handle)

checks = {
    "seis canales": len(reloaded["channels"]) == 6,
    "corte de censura correcto": reloaded["meta"]["censoring_cutoff"] == "2026-05-31",
    "serie de CAC aparente": len(reloaded["censoring"]["apparent_cac_by_month"]) > 24,
    "el tramo censurado está marcado": any(
        m["is_censored"] for m in reloaded["censoring"]["apparent_cac_by_month"]),
    "caída del CAC aparente > 70%": reloaded["censoring"]["drop_pct"] < -70,
    "código compartido con dos creadores": len(reloaded["shared_influencer_code"]["creators"]) == 2,
    "el coste ponderado es la mitad del bruto": abs(
        reloaded["shared_influencer_code"]["weighted_cost_eur"] * 2
        - reloaded["shared_influencer_code"]["gross_cost_eur"]) < 1.0,
    "iOS ATT sólo afecta a paid_social": (
        reloaded["ios_att"]["paid_social_change_pct"] < -50
        and abs(reloaded["ios_att"]["other_channels_inside_pct"]
                - reloaded["ios_att"]["other_channels_outside_pct"]) < 2),
    "las dos mitades del gap": (
        reloaded["attribution_gap"]["orphan_touchpoints"] > 0
        and reloaded["attribution_gap"]["conversions_without_touchpoint"] > 0),
    "el CAC cargado supera al atribuible": (
        reloaded["attribution_gap"]["cac_loaded_eur"]
        > reloaded["attribution_gap"]["cac_attributable_eur"]),
    "cuatro modelos con crédito": all(
        {"first_touch", "last_touch", "linear", "markov"} <= set(row)
        for row in reloaded["models"]["credit_pct"]),
    "cada modelo reparte el 100%": all(
        abs(sum(row[m] for row in reloaded["models"]["credit_pct"]) - 100) < 0.01
        for m in ("first_touch", "last_touch", "linear", "markov")),
    "markov reproduce la conversión observada": abs(
        reloaded["markov"]["model_conversion_probability"]
        - reloaded["markov"]["observed_conversion_rate"]) < 0.01,
    "matriz de transición cuadrada": all(
        len(row) == len(reloaded["markov"]["states"]) + 1
        for row in reloaded["markov"]["transition_matrix_pct"]),
    "filas de la matriz suman 100": all(
        abs(sum(v for k, v in row.items() if k != "from") - 100) < 0.01
        for row in reloaded["markov"]["transition_matrix_pct"]),
    "removal effects positivos": all(
        v > 0 for v in reloaded["markov"]["removal_effect_pct"].values()),
    "CAC por canal": len(reloaded["cac"]["by_channel"]) == len(reloaded["models"]["credit_pct"]),
    "el CAC medio del reparto cuadra con el atribuible": abs(
        reloaded["cac"]["blended_markov_eur"]
        - reloaded["attribution_gap"]["cac_attributable_eur"]) < 0.01,
    "el CAC cargado medio supera al del reparto": (
        reloaded["cac"]["blended_loaded_eur"] > reloaded["cac"]["blended_markov_eur"]),
    "el coste por canal cuadra con el total": abs(
        sum(c["total_cost_eur"] for c in reloaded["channels"])
        - (reloaded["attribution_gap"]["orphan_cost_eur"]
           + sum(c["attributable_cost_eur"] for c in reloaded["channels"]))) < 1.0,
    "el CAC cargado por canal es mayor": all(
        row["cac_cargado_markov"] is None or row["markov"] is None
        or row["cac_cargado_markov"] >= row["markov"] - 1e-9
        for row in reloaded["cac"]["by_channel"]),
}
for label, ok in checks.items():
    print(f"  {'OK ' if ok else 'FALLO'} {label}")
assert all(checks.values()), "El JSON de salida no tiene la forma esperada."
print()
print("JSON verificado.")

  OK  seis canales
  OK  corte de censura correcto
  OK  serie de CAC aparente
  OK  el tramo censurado está marcado
  OK  caída del CAC aparente > 70%
  OK  código compartido con dos creadores
  OK  el coste ponderado es la mitad del bruto
  OK  iOS ATT sólo afecta a paid_social
  OK  las dos mitades del gap
  OK  el CAC cargado supera al atribuible
  OK  cuatro modelos con crédito
  OK  cada modelo reparte el 100%
  OK  markov reproduce la conversión observada
  OK  matriz de transición cuadrada
  OK  filas de la matriz suman 100
  OK  removal effects positivos
  OK  CAC por canal
  OK  el CAC medio del reparto cuadra con el atribuible
  OK  el CAC cargado medio supera al del reparto
  OK  el coste por canal cuadra con el total
  OK  el CAC cargado por canal es mayor

JSON verificado.


## Conclusiones

1. **Antes de repartir el mérito hay que arreglar el denominador.** Los recorridos están construidos
   hacia atrás desde el alta, así que el tramo final del histórico tiene touchpoints a medias y el
   CAC aparente del último mes sale un 79% por debajo del de enero. Es la conclusión más golosa que
   puede producir esta página y es falsa; el corte a 31 de mayo de 2026 es la primera decisión, no
   la última.
2. **El attribution gap son dos huecos y el que se cita es el pequeño.** El 6,2% de las altas sin
   touchpoint es el número que pide el encargo; el que mueve el dinero es el otro, el 39,4% de
   touchpoints sin cliente que se lleva el 24,5% del gasto. Juntos hacen que el CAC atribuible
   (11,71 €) infravalore el real (14,53 €) **un 24%**.
3. **Y el gap no es neutral entre canales.** Afecta al 39% de los touchpoints de `paid_social` y al
   22% de los de `referral`, así que ignorarlo no baja el CAC de todos por igual: cambia el orden
   relativo, que es justo lo que una comparación de canales pretende medir.
4. **El coste hay que ponderarlo antes de dividirlo.** Dos creadores reclaman el mismo código y
   sumar el coste sin más mete 1.152 € inexistentes en el canal de influencer. Se arregla en el
   numerador, no en el modelo: el cliente es uno, lo duplicado es el registro del gasto.
5. **Hay imperfecciones que no se arreglan, sólo se declaran.** La pérdida de trazabilidad de
   `paid_social` entre enero y junio de 2025 le quita conversiones que sí ocurrieron. No hay forma
   de recuperarlas; lo único honesto es marcar el tramo y decir que su CAC publicado es peor que el
   real.
6. **El modelo de atribución era la pregunta menos importante.** Los cuatro reparten el crédito con
   una dispersión máxima de 3,6 puntos y ordenan los canales igual, porque el recorrido medio tiene
   2,48 contactos. Elegir modelo mueve el CAC un 3%; tratar el gap lo mueve un 24%. La conclusión
   sólo vale porque se ha medido: en un negocio con recorridos largos, la respuesta sería la
   contraria.
7. **Y el modelo sofisticado hay que implementarlo bien o no mide nada.** Calcular el removal effect
   borrando el canal de la secuencia, en vez de redirigir a no-conversión lo que pasa por él,
   devuelve cero para todos los canales y una tabla con el aspecto de siempre.